In [3]:
import pandas as pd

In [4]:
ISOLAR_PATH = "./data/isolar.txt"
R_PATH = "./data/20%-2.txt"
separator = "\t"

In [5]:
# def calculate(isolar_path, r_path, lower = 300, upper = 2400, separator='\t'):

In [6]:
isolar_path = ISOLAR_PATH
r_path = R_PATH
lower = 300
upper = 2400
separator = "\t"

In [7]:
import pandas as pd
# 读取数据
isolar = pd.read_csv(isolar_path, sep=separator, header=None, names=['wavelength_nm', 'I_solar'])
r = pd.read_csv(r_path, sep=separator, header=None, names=['wavelength_nm', 'R'])

# 过滤掉小数点后有数字的行
isolar_int = isolar[isolar['wavelength_nm'] % 1 == 0]
r_int = r[r['wavelength_nm'] % 1 == 0]

isolar_filtered_int = isolar_int[(isolar_int['wavelength_nm'] >= lower) & (isolar_int['wavelength_nm'] <= upper)]
r_filtered_int = r_int[(r_int['wavelength_nm'] >= lower) & (r_int['wavelength_nm'] <= upper)]

# 明确创建副本
isolar_filtered_int = isolar_int[(isolar_int['wavelength_nm'] >= lower) & (isolar_int['wavelength_nm'] <= upper)].copy()
r_filtered_int = r_int[(r_int['wavelength_nm'] >= lower) & (r_int['wavelength_nm'] <= upper)].copy()

isolar_filtered_int['wavelength_um'] = isolar_filtered_int['wavelength_nm'] / 1000
r_filtered_int['wavelength_um'] = r_filtered_int['wavelength_nm'] / 1000

# Restrict to common range
isolar_range = isolar_filtered_int[(isolar_filtered_int['wavelength_um'] >= 0.5) & 
                                (isolar_filtered_int['wavelength_um'] <= 2.4)]
r_range = r_filtered_int[(r_filtered_int['wavelength_um'] >= 0.5) & 
                        (r_filtered_int['wavelength_um'] <= 2.4)]

# Merge on common wavelength values
merged = pd.merge(isolar_range[['wavelength_um', 'I_solar']], 
                r_range[['wavelength_um', 'R']], 
                on='wavelength_um')

# ------------------- Handle the missing value -------------------
merged = merged.dropna()
merged['I_solar'] = merged['I_solar'].astype(float)
merged['R'] = merged['R'].astype(float)
merged['wavelength_um'] = merged['wavelength_um'].astype(float)

# --------------------Handle the missing value end-------------------


# Convert R from percent to 0-1
merged['R'] /= 100

# Assume uniform delta_lambda = 0.001 μm
delta_lambda = 0.001

# Compute weighted reflectance
numerator = (merged['I_solar'] * merged['R']).sum() * delta_lambda
denominator = merged['I_solar'].sum() * delta_lambda
R_solar_discrete = numerator / denominator

# 保留4位小数
R_solar_discrete = round(R_solar_discrete, 6)

R_solar_discrete

np.float64(0.93303)

In [ ]:
import numpy as np

# 使用梯形法计算积分
# 对 merged['I_solar'] * merged['R'] 进行积分，分母对 merged['I_solar'] 积分

# 计算分子和分母的积分
numerator_trapz = np.trapezoid(merged['I_solar'] * merged['R'], merged['wavelength_um'])
denominator_trapz = np.trapezoid(merged['I_solar'], merged['wavelength_um'])

R_solar_trapz = numerator_trapz / denominator_trapz
R_solar_trapz = round(R_solar_trapz, 6)
R_solar_trapz

np.float64(0.925262)